In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, when

spark = SparkSession.builder \
    .appName("WarehouseInsights") \
    .getOrCreate()
print("Spark Session created")

Spark Session created


In [3]:
data = [
    (1, "Wireless Mouse", 1, "Warehouse Alpha", "IN", 120),
    (1, "Wireless Mouse", 2, "Warehouse Beta", "IN", 80),
    (1, "Wireless Mouse", 3, "Warehouse Gamma", "IN", 60),
    (2, "Mechanical Keyboard", 1, "Warehouse Alpha", "IN", 50),
    (2, "Mechanical Keyboard", 2, "Warehouse Beta", "OUT", 42),
    (2, "Mechanical Keyboard", 3, "Warehouse Gamma", "OUT", 30),
    (3, "USB-C Hub", 1, "Warehouse Alpha", "IN", 45),
    (3, "USB-C Hub", 2, "Warehouse Beta", "IN", 30),
    (3, "USB-C Hub", 3, "Warehouse Gamma", "OUT", 25),
    (4, "Laptop Stand", 1, "Warehouse Alpha", "IN", 400),
    (4, "Laptop Stand", 2, "Warehouse Beta", "IN", 380),
    (5, "HDMI Cable 2m", 1, "Warehouse Alpha", "OUT", 25),
    (5, "HDMI Cable 2m", 2, "Warehouse Beta", "OUT", 8),
    (6, "Webcam 1080p", 1, "Warehouse Alpha", "IN", 24),
    (6, "Webcam 1080p", 2, "Warehouse Beta", "ADJUSTMENT", 20),
    (7, "Desk Lamp LED", 1, "Warehouse Alpha", "IN", 55),
    (7, "Desk Lamp LED", 2, "Warehouse Beta", "IN", 70),
    (7, "Desk Lamp LED", 3, "Warehouse Gamma", "IN", 90),
    (8, "Ergonomic Chair", 1, "Warehouse Alpha", "IN", 3),
    (8, "Ergonomic Chair", 2, "Warehouse Beta", "OUT", 1),
]

schema = "product_id INT, product_name STRING, warehouse_id INT, warehouse_name STRING, movement_type STRING, quantity INT"
df = spark.createDataFrame(data, schema=schema)
print("Data loaded into Spark")
df.show(20)

Data loaded into Spark
+----------+-------------------+------------+---------------+-------------+--------+
|product_id|       product_name|warehouse_id| warehouse_name|movement_type|quantity|
+----------+-------------------+------------+---------------+-------------+--------+
|         1|     Wireless Mouse|           1|Warehouse Alpha|           IN|     120|
|         1|     Wireless Mouse|           2| Warehouse Beta|           IN|      80|
|         1|     Wireless Mouse|           3|Warehouse Gamma|           IN|      60|
|         2|Mechanical Keyboard|           1|Warehouse Alpha|           IN|      50|
|         2|Mechanical Keyboard|           2| Warehouse Beta|          OUT|      42|
|         2|Mechanical Keyboard|           3|Warehouse Gamma|          OUT|      30|
|         3|          USB-C Hub|           1|Warehouse Alpha|           IN|      45|
|         3|          USB-C Hub|           2| Warehouse Beta|           IN|      30|
|         3|          USB-C Hub|          

In [4]:
stock_in = df.filter(col("movement_type") == "IN").groupBy("warehouse_id", "warehouse_name").agg(sum("quantity").alias("stock_in"))
stock_out = df.filter(col("movement_type") == "OUT").groupBy("warehouse_id", "warehouse_name").agg(sum("quantity").alias("stock_out"))
adjustments = df.filter(col("movement_type") == "ADJUSTMENT").groupBy("warehouse_id", "warehouse_name").agg(sum("quantity").alias("adjustments"))
warehouse_agg = stock_in.join(stock_out, on=["warehouse_id", "warehouse_name"], how="outer").join(adjustments, on=["warehouse_id", "warehouse_name"], how="outer")
warehouse_agg = warehouse_agg.fillna(0)
warehouse_agg = warehouse_agg.withColumn("current_stock", col("stock_in") - col("stock_out") + col("adjustments"))
print("Warehouse Aggregation:")
warehouse_agg.show()

Warehouse Aggregation:
+------------+---------------+--------+---------+-----------+-------------+
|warehouse_id| warehouse_name|stock_in|stock_out|adjustments|current_stock|
+------------+---------------+--------+---------+-----------+-------------+
|           1|Warehouse Alpha|     697|       25|          0|          672|
|           2| Warehouse Beta|     560|       51|         20|          529|
|           3|Warehouse Gamma|     150|       55|          0|           95|
+------------+---------------+--------+---------+-----------+-------------+



In [5]:
avg_stock = warehouse_agg.agg({"current_stock": "avg"}).collect()[0][0]
overstock_threshold = avg_stock * 1.5
understock_threshold = avg_stock * 0.5
print(f"Average stock per warehouse: {avg_stock}")
print(f"Overstock threshold: {overstock_threshold}")
print(f"Understock threshold: {understock_threshold}")
warehouse_status = warehouse_agg.withColumn(
    "status",
    when(col("current_stock") >= overstock_threshold, "OVERSTOCKED")
    .when(col("current_stock") <= understock_threshold, "UNDERSTOCKED")
    .otherwise("NORMAL")
)
print("\nWarehouse Stock Status:")
warehouse_status.show()

Average stock per warehouse: 432.0
Overstock threshold: 648.0
Understock threshold: 216.0

Warehouse Stock Status:
+------------+---------------+--------+---------+-----------+-------------+------------+
|warehouse_id| warehouse_name|stock_in|stock_out|adjustments|current_stock|      status|
+------------+---------------+--------+---------+-----------+-------------+------------+
|           1|Warehouse Alpha|     697|       25|          0|          672| OVERSTOCKED|
|           2| Warehouse Beta|     560|       51|         20|          529|      NORMAL|
|           3|Warehouse Gamma|     150|       55|          0|           95|UNDERSTOCKED|
+------------+---------------+--------+---------+-----------+-------------+------------+



In [6]:
overstocked= warehouse_status.filter(col("status") == "OVERSTOCKED")
understocked= warehouse_status.filter(col("status") == "UNDERSTOCKED")
print("Overstocked Warehouses:")
overstocked.show()
print("\nUnderstocked Warehouses:")
understocked.show()

Overstocked Warehouses:
+------------+---------------+--------+---------+-----------+-------------+-----------+
|warehouse_id| warehouse_name|stock_in|stock_out|adjustments|current_stock|     status|
+------------+---------------+--------+---------+-----------+-------------+-----------+
|           1|Warehouse Alpha|     697|       25|          0|          672|OVERSTOCKED|
+------------+---------------+--------+---------+-----------+-------------+-----------+


Understocked Warehouses:
+------------+---------------+--------+---------+-----------+-------------+------------+
|warehouse_id| warehouse_name|stock_in|stock_out|adjustments|current_stock|      status|
+------------+---------------+--------+---------+-----------+-------------+------------+
|           3|Warehouse Gamma|     150|       55|          0|           95|UNDERSTOCKED|
+------------+---------------+--------+---------+-----------+-------------+------------+



In [7]:
summary = warehouse_status.groupBy("status").agg(count("warehouse_id").alias("count"))
print("Stock Status Summary:")
summary.show()

Stock Status Summary:
+------------+-----+
|      status|count|
+------------+-----+
| OVERSTOCKED|    1|
|UNDERSTOCKED|    1|
|      NORMAL|    1|
+------------+-----+



In [8]:
warehouse_status.coalesce(1).write.mode("overwrite").csv("/content/warehouse_status_report", header=True)
print("Report saved to warehouse_status_report")
warehouse_status.show(truncate=False)

Report saved to warehouse_status_report
+------------+---------------+--------+---------+-----------+-------------+------------+
|warehouse_id|warehouse_name |stock_in|stock_out|adjustments|current_stock|status      |
+------------+---------------+--------+---------+-----------+-------------+------------+
|1           |Warehouse Alpha|697     |25       |0          |672          |OVERSTOCKED |
|2           |Warehouse Beta |560     |51       |20         |529          |NORMAL      |
|3           |Warehouse Gamma|150     |55       |0          |95           |UNDERSTOCKED|
+------------+---------------+--------+---------+-----------+-------------+------------+



In [9]:
result_df = warehouse_status.select("warehouse_id", "warehouse_name", "current_stock", "status").toPandas()
print(result_df.to_string())
result_df.to_csv('warehouse_stock_status.csv', index=False)
print("\nCSV saved as warehouse_stock_status.csv")

   warehouse_id   warehouse_name  current_stock        status
0             1  Warehouse Alpha            672   OVERSTOCKED
1             2   Warehouse Beta            529        NORMAL
2             3  Warehouse Gamma             95  UNDERSTOCKED

CSV saved as warehouse_stock_status.csv
